# Pipeline: Razonamiento, Búsqueda y Resultados
## Caso Tipo B: IRA y Neumonía en menores de 5 años (Perú)

**Challenge 1 (Individual): Pipeline de Machine Learning, UPCH**

Este notebook representa el razonamiento que seguiría una IA para decidir **qué combinación de intervenciones priorizar** y cumple la estructura pedida:

1. **Problema** · 2. **Espacio de Estados** · 3. **Exploración** · 4. **Búsqueda** (BFS, UCS, A\*) · 5. **Resultado**

> **Aviso de supuestos:** la infografía no reporta costos ni el impacto individual de cada intervención. Los factores de reducción y costos de este notebook son **hipótesis didácticas**, no datos oficiales.

## 1. Problema

Con recursos limitados, ¿qué combinación de intervenciones logra una **reducción adicional del 15 %** de los episodios de IRA (de 824 978 a ≤ 701 231) al **menor costo** posible, sin superar un presupuesto de 12 unidades relativas?

In [ ]:
import heapq, math
from collections import deque
from itertools import combinations
import matplotlib.pyplot as plt
import pandas as pd

BASE = 824978                 # episodios de IRA en 2025 (anexo)
META_RED = 0.15               # meta: reducir 15 % adicional
TARGET = BASE * (1 - META_RED)
PRESUPUESTO = 12              # supuesto: costos no reportados en el anexo

# (factor de riesgo residual, costo relativo)  -> SUPUESTOS didacticos
INTERV = {
    "Vacuna neumococo":  (0.94, 5),
    "Vacuna influenza":  (0.95, 4),
    "Lavado de manos":   (0.96, 1),
    "Mascarilla":        (0.97, 1),
    "Ventilacion":       (0.97, 2),
    "Atencion oportuna": (0.98, 3),
}
NAMES = list(INTERV)
N = len(NAMES)
print(f"Episodios base: {BASE:,} | Meta: <= {TARGET:,.0f} | Presupuesto: {PRESUPUESTO}")

## 2. Espacio de Estados

- **Estado:** conjunto de intervenciones activas (con sus episodios estimados y costo acumulado).
- **Estado inicial:** ninguna intervención activa (824 978 episodios, costo 0).
- **Operadores:** `activar(k)`, una intervención aún no activa. Su efecto es multiplicativo.
- **Objetivo:** episodios ≤ 701 231.
- **Restricciones:** cada intervención se activa una sola vez y el costo acumulado no puede superar el presupuesto.

In [ ]:
def episodios(state):
    r = BASE
    for i in state:
        r *= INTERV[NAMES[i]][0]
    return r

def costo(state):
    return sum(INTERV[NAMES[i]][1] for i in state)

def es_meta(state):
    return episodios(state) <= TARGET

def vecinos(state):
    """Operadores aplicables: activar una intervencion no activa, respetando el presupuesto."""
    for i in range(N):
        if i not in state:
            nuevo = frozenset(state | {i})
            if costo(nuevo) <= PRESUPUESTO:
                yield i, nuevo

todos = [frozenset(cmb) for k in range(N + 1) for cmb in combinations(range(N), k)]
validos = [s for s in todos if costo(s) <= PRESUPUESTO]
print(f"Estados totales: {len(todos)} | dentro del presupuesto: {len(validos)}")

## 3. Exploración

Antes de buscar con una estrategia concreta se **explora todo el espacio**: ¿qué estados cumplen la meta y a qué costo?

In [ ]:
filas = []
for s in validos:
    filas.append({
        "Intervenciones": ", ".join(NAMES[i] for i in sorted(s)) or "(ninguna)",
        "N°": len(s), "Costo": costo(s),
        "Episodios": round(episodios(s)),
        "Reduccion %": round(100 * (1 - episodios(s) / BASE), 1),
        "Cumple meta": es_meta(s),
    })
df = pd.DataFrame(filas)
print("Estados que cumplen la meta (ordenados por costo):")
df[df["Cumple meta"]].sort_values("Costo")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ok = df[df["Cumple meta"]]; no = df[~df["Cumple meta"]]
ax.scatter(no["Costo"], no["Reduccion %"], color="#0E7C7B", alpha=0.7, label="No cumple la meta")
ax.scatter(ok["Costo"], ok["Reduccion %"], color="#2E9E5B", s=80, label="Cumple la meta")
ax.axhline(15, color="#B0201B", ls="--", label="Meta: 15 %")
ax.set_xlabel("Costo relativo acumulado"); ax.set_ylabel("Reducción de episodios de IRA (%)")
ax.set_title("Exploración del espacio de estados: costo vs. reducción"); ax.legend()
plt.show()

## 4. Búsqueda

### 4.1 BFS (no informada)
Explora por niveles (menor número de intervenciones). **No garantiza el menor costo.**

In [ ]:
def bfs():
    ini = frozenset()
    fr = deque([(ini, [])]); vis = {ini}; exp = 0
    while fr:
        s, p = fr.popleft(); exp += 1
        if es_meta(s): return p, s, exp
        for i, n in vecinos(s):
            if n not in vis:
                vis.add(n); fr.append((n, p + [i]))

p_bfs, s_bfs, e_bfs = bfs()
print("BFS ->", [NAMES[i] for i in p_bfs], "| costo", costo(s_bfs), "| episodios", round(episodios(s_bfs)), "| nodos expandidos", e_bfs)

### 4.2 UCS (no informada, costo uniforme)
Expande siempre el estado de **menor costo acumulado**, por lo que garantiza el costo mínimo.

In [ ]:
def busqueda_costo(informada=False):
    ini = frozenset(); cnt = 0
    fr = [(h(ini) if informada else 0, 0, cnt, ini, [])]; vis = set(); exp = 0
    while fr:
        f, g, _, s, p = heapq.heappop(fr)
        if s in vis: continue
        vis.add(s); exp += 1
        if es_meta(s): return p, s, exp
        for i, n in vecinos(s):
            if n not in vis:
                cnt += 1; g2 = costo(n)
                heapq.heappush(fr, (g2 + (h(n) if informada else 0), g2, cnt, n, p + [i]))

def h(state):  # placeholder; se redefine en 4.3 para A*
    return 0

p_ucs, s_ucs, e_ucs = busqueda_costo(False)
print("UCS ->", [NAMES[i] for i in p_ucs], "| costo", costo(s_ucs), "| episodios", round(episodios(s_ucs)), "| nodos expandidos", e_ucs)

### 4.3 A\* (informada)

**Heurística:** costo mínimo *teórico* restante, calculado en escala logarítmica con la mejor relación efecto/costo disponible (relajación fraccional). Nunca sobreestima el costo real, por lo que es **admisible** y A\* mantiene la optimalidad.

In [ ]:
RATE = [-math.log(INTERV[n][0]) for n in NAMES]                 # reduccion logaritmica de cada intervencion
BEST_RATIO = max(RATE[i] / INTERV[NAMES[i]][1] for i in range(N)) # mejor efecto por unidad de costo

def h(state):
    falta = math.log(episodios(state) / TARGET)
    return max(0.0, falta) / BEST_RATIO

p_as, s_as, e_as = busqueda_costo(True)
print("A*  ->", [NAMES[i] for i in p_as], "| costo", costo(s_as), "| episodios", round(episodios(s_as)), "| nodos expandidos", e_as)

## 5. Resultado

In [ ]:
res = pd.DataFrame([
    {"Estrategia": "BFS", "Intervenciones": " + ".join(NAMES[i] for i in p_bfs), "Costo": costo(s_bfs), "Episodios": round(episodios(s_bfs)), "Nodos expandidos": e_bfs},
    {"Estrategia": "UCS", "Intervenciones": " + ".join(NAMES[i] for i in p_ucs), "Costo": costo(s_ucs), "Episodios": round(episodios(s_ucs)), "Nodos expandidos": e_ucs},
    {"Estrategia": "A*",  "Intervenciones": " + ".join(NAMES[i] for i in p_as),  "Costo": costo(s_as),  "Episodios": round(episodios(s_as)),  "Nodos expandidos": e_as},
])
res

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(res["Estrategia"], res["Nodos expandidos"], color=["#0E7C7B", "#0B3D5C", "#E8842C"])
axes[0].set_title("Nodos expandidos (eficiencia de búsqueda)"); axes[0].set_ylabel("Nodos")
axes[1].bar(res["Estrategia"], res["Costo"], color=["#0E7C7B", "#0B3D5C", "#E8842C"])
axes[1].set_title("Costo de la solución encontrada"); axes[1].set_ylabel("Costo relativo")
plt.tight_layout(); plt.show()

# Trayectoria de la ruta optima: episodios estimados paso a paso
pasos = [BASE]; etiquetas = ["Inicio"]; est = frozenset()
for i in p_as:
    est = frozenset(est | {i}); pasos.append(episodios(est)); etiquetas.append("+ " + NAMES[i])
plt.figure(figsize=(9, 4.5))
plt.plot(etiquetas, pasos, marker="o", color="#0B3D5C", lw=2)
plt.axhline(TARGET, color="#B0201B", ls="--", label=f"Meta: {TARGET:,.0f}")
plt.xticks(rotation=20); plt.ylabel("Episodios de IRA estimados")
plt.title("Ruta óptima: reducción de episodios al activar cada intervención"); plt.legend(); plt.show()

### Interpretación

- **BFS** cumple la meta con el menor número de intervenciones (4), pero a un costo de 11, mayor que el óptimo.
- **UCS** y **A\*** encuentran el **costo mínimo (9)**: lavado de manos, mascarilla, ventilación y vacuna contra neumococo. Esto sugiere que las medidas baratas (higiene y ventilación) deben activarse antes que las más costosas.
- **A\*** llega a la misma solución expandiendo **menos nodos** que UCS gracias a su heurística admisible.
- **Límites:** los factores de efecto y costo son supuestos. La infografía señala que el impacto individual de cada medida, los costos y la relación causal **no están reportados**; con datos reales, la ruta óptima podría cambiar.